# Kaggle Claims Ingestion

## Purpose

This notebook ingests the synthetic health insurance claims dataset
into the Bronze layer of the Khaoula Healthy Insurance Data Platform.

The Bronze layer preserves source data with minimal transformation
while adding ingestion metadata for traceability.

### Source
- Synthetic health insurance claims CSV
- 20,100 records
- 30 source columns

### Target
`health_insurance.bronze.claims_raw`



In [0]:
CATALOG = 'health_insurance'
BRONZE_SCHEMA = 'bronze'

TARGET_TABLE = f'{CATALOG}.{BRONZE_SCHEMA}.claims_raw'

print(f'Target Bronze table: {TARGET_TABLE}')

In [0]:
%sql
-- ============================================================
-- Create Bronze raw-file Volume
-- ============================================================

CREATE VOLUME IF NOT EXISTS health_insurance.bronze.raw_files
COMMENT 'Landing area for raw source files used by the health insurance platform';

-- VERIFY CREATION

SHOW VOLUMES IN health_insurance.bronze;

In [0]:
# ============================================================
# Source file configuration
# ============================================================

SOURCE_FILE = (
    "/Volumes/health_insurance/bronze/raw_files/"
    "synthetic_health_claims.csv"
)

print(f"Source file: {SOURCE_FILE}")

In [0]:
# ============================================================
# Read raw claims CSV
# ============================================================

claims_raw_df = (
    spark.read
        .option('header', True)
        .option('inferSchema', True)
        .csv(SOURCE_FILE)
)

display(claims_raw_df.limit(10))

In [0]:
# ============================================================
# Validate source dimensions
# ============================================================

row_count = claims_raw_df.count()
column_count = len(claims_raw_df.columns)

print(f"Rows: {row_count:,}")
print(f"Columns: {column_count}")

In [0]:
# ============================================================
# Inspect inferred Spark schema
# ============================================================

claims_raw_df.printSchema()

In [0]:
# ============================================================
# Inspect source column names
# ============================================================

for position, column_name in enumerate(claims_raw_df.columns, start=1):
    print(f"{position:02d}. {column_name}")

In [0]:
# ============================================================
# Inspect duplicate Claim_ID values
# ============================================================

duplicate_claim_ids = (
    claims_raw_df
    .groupBy("Claim_ID")
    .count()
    .filter("count > 1")
)

print(
    "Duplicate Claim_ID groups:",
    duplicate_claim_ids.count()
)

display(duplicate_claim_ids.limit(20))

In [0]:
# ============================================================
# Inspect null critical identifiers
# ============================================================

from pyspark.sql import functions as F

claims_raw_df.select(
    F.sum(F.col("Patient_ID").isNull().cast("int")).alias("null_patient_ids"),
    F.sum(F.col("Policy_Number").isNull().cast("int")).alias("null_policy_numbers"),
    F.sum(F.col("Claim_ID").isNull().cast("int")).alias("null_claim_ids")
).show()

In [0]:
# ============================================================
# Add ingestion metadata
# ============================================================

from pyspark.sql import functions as F

claims_bronze_df = (
    claims_raw_df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_system", F.lit("kaggle"))
    .withColumn("_source_file", F.lit("synthetic_health_claims.csv"))
)

display(claims_bronze_df.limit(5))

In [0]:
# ============================================================
# Persist raw claims into Bronze Delta
# ============================================================

(
    claims_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

print(f"Created Bronze table: {TARGET_TABLE}")

In [0]:
%sql
SELECT *
FROM health_insurance.bronze.claims_raw
LIMIT 10;

## Ingestion Result

The Kaggle synthetic health insurance claims dataset was successfully
ingested into the Bronze layer.

### Source
`synthetic_health_claims.csv`

### Target
`health_insurance.bronze.claims_raw`

### Bronze design principle

The Bronze layer preserves the original source columns and adds only
technical ingestion metadata:

- `_ingested_at`
- `_source_system`
- `_source_file`

No business cleaning or standardization is performed at this stage.
Those transformations will occur in the Silver layer.